<a href="https://colab.research.google.com/github/fcoliveira-utfpr/climate_parana/blob/main/01_area_of_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Start – Libraries
---

In [ ]:
# -------------------------
# IMPORTS
# -------------------------
!pip install geobr geopandas rasterio elevation cartopy --quiet
import geobr
import geopandas as gpd
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.0/338.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.3 MB/s eta 0:00:00


# Data source
---

In [ ]:
# ----------------------------
# Data Loading (geobr)
# ----------------------------
# Municipalities of Paraná
mun_pr = geobr.read_municipality(code_muni="PR", year=2020)

# Biomes
biomas = geobr.read_biomes(year=2019)

# Conservation Units
ucs = geobr.read_conservation_units()

# Indigenous Lands
ti = geobr.read_indigenous_land()

# ----------------------------
# Appropriate projection for area calculation
# ----------------------------
epsg_proj = 31982  # SIRGAS 2000 / UTM zone 22S (Paraná)

mun_pr = mun_pr.to_crs(epsg_proj)
biomas = biomas.to_crs(epsg_proj)
ucs = ucs.to_crs(epsg_proj)
ti = ti.to_crs(epsg_proj)

# ----------------------------
# Municipality area (km²)
# ----------------------------
mun_pr["area_km2"] = mun_pr.geometry.area / 1e6

# ----------------------------
# Dominant biome per municipality
# ----------------------------
inter_bioma = gpd.overlay(mun_pr, biomas, how="intersection")
inter_bioma["area_intersec"] = inter_bioma.geometry.area

bioma_dom = (
    inter_bioma
    .sort_values("area_intersec", ascending=False)
    .drop_duplicates("code_muni")[["code_muni", "name_biome"]]
)

mun_pr = mun_pr.merge(bioma_dom, on="code_muni", how="left")

# ----------------------------
# Presence of Conservation Units and Indigenous Lands
# ----------------------------
mun_pr["has_conservation_unit"] = mun_pr.intersects(ucs.union_all())
mun_pr["has_indigenous_land"] = mun_pr.intersects(ti.union_all())

# ----------------------------
# Final result as DataFrame
# ----------------------------
df_parana_environment_rural = mun_pr.drop(columns="geometry")

# Preview
df_parana_environment_rural.head()

,code_muni,name_muni,code_state,abbrev_state,name_state,code_region,name_region,area_km2,name_biome,has_conservation_unit,has_indigenous_land
0,4100103.0,Abatiá,41.0,PR,Paraná,4.0,Sul,228.518093,Mata Atlântica,False,True
1,4100202.0,Adrianópolis,41.0,PR,Paraná,4.0,Sul,1350.483016,Mata Atlântica,True,False
2,4100301.0,Agudos Do Sul,41.0,PR,Paraná,4.0,Sul,192.357285,Mata Atlântica,False,False
3,4100400.0,Almirante Tamandaré,41.0,PR,Paraná,4.0,Sul,194.757763,Mata Atlântica,True,False
4,4100459.0,Altamira Do Paraná,41.0,PR,Paraná,4.0,Sul,387.010301,Mata Atlântica,False,False


# Summarized data
---

In [ ]:
# ============================================================
# SUMMARY DATAFRAME – ENVIRONMENT & TERRITORY (PARANÁ)
# ============================================================

# Number of municipalities
n_mun = mun_pr.shape[0]

# Total state area (km²)
area_total_estado_km2 = mun_pr["area_km2"].sum()

# Average municipality area
area_media_municipios_km2 = mun_pr["area_km2"].mean()

# Municipalities by dominant biome
biomas_mun = (
    mun_pr["name_biome"]
    .value_counts()
    .to_dict()
)

# Percentage of municipalities with Conservation Units
perc_mun_uc = mun_pr["has_conservation_unit"].mean() * 100

# Percentage of municipalities with Indigenous Lands
perc_mun_ti = mun_pr["has_indigenous_land"].mean() * 100

# ----------------------------
# Final DataFrame
# ----------------------------
df_resumo_parana = pd.DataFrame({
    "state": ["Paraná"],
    "number_of_municipalities": [n_mun],
    "total_area_km2": [area_total_estado_km2],
    "average_municipality_area_km2": [area_media_municipios_km2],
    "percentage_municipalities_with_conservation_units": [perc_mun_uc],
    "percentage_municipalities_with_indigenous_lands": [perc_mun_ti],
    "municipalities_by_biome": [biomas_mun]
})

df_resumo_parana

,state,number_of_municipalities,total_area_km2,average_municipality_area_km2,percentage_municipalities_with_conservation_units,percentage_municipalities_with_indigenous_lands,municipalities_by_biome
0,Paraná,399,199273.987969,499.433554,24.81203,9.273183,"{'Mata Atlântica': 398, 'Cerrado': 1}"


#Climate zoning by municipality

In [ ]:
# =========================================================
# DADOS – MAPA ESQUERDO (MUNICÍPIOS)
# =========================================================
url = "https://raw.githubusercontent.com/fcoliveira-utfpr/climate_parana/refs/heads/main/dados_concatenados_zc.csv"
df = pd.read_csv(url, sep=",")

df['variavel'] = df['variavel'].str.lower()
df

,estado,lat,lon,municipio,valor,variavel,ZC
0,Parana,-23.311363,-50.306176,ABATIA,210.718086,def,7601
1,Parana,-25.884047,-53.640829,BELA VISTA DA CAROBA,87.526841,def,6801
2,Parana,-24.781986,-48.819381,ADRIANOPOLIS,54.045608,def,6801
3,Parana,-26.025501,-49.309155,AGUDOS DO SUL,33.173108,def,6801
4,Parana,-25.284879,-49.322794,ALMIRANTE TAMANDARE,35.327444,def,6801
...,...,...,...,...,...,...,...
2401,Parana,-23.909392,-50.067200,PINHALAO,1121.243333,pet,6601
2402,Parana,-23.909392,-50.067200,PINHALAO,1307.033333,pr,6601
2403,Parana,-23.909392,-50.067200,PINHALAO,325.533333,ro,6601
2404,Parana,-23.909392,-50.067200,PINHALAO,14.104444,tmmn,6601


In [ ]:
df.to_excel('dados_concatenados_zc.xlsx')

In [ ]:
zc_info_dict = {
    5901: {"regiao": "South",
           "area": 5430.239963591397},
    6601: {"regiao": "Central-South",
           "area": 221.12865423926547},
    6701: {"regiao": "Central-South",
           "area": 16002.214916721656},
    6801: {"regiao": "Central-South and Metropolitan Region of Curitiba",
           "area": 75856.20194971134},
    6901: {"regiao": "Southwest and South",
           "area": 9525.499533375816},
    7501: {"regiao": "Northwest",
           "area": 7211.771280980986},
    7601: {"regiao": "North and North Pioneer",
           "area": 23114.615047292868},
    7701: {"regiao": "West and North Central",
           "area": 39302.6450045043},
    7801: {"regiao": "West",
           "area": 17702.602407614897},
    7901: {"regiao": "Coastal",
           "area": 5381.529570849405}
}

In [ ]:
#Numero de município por ZC
resultado = df.groupby('ZC').agg(
    num_municipios=('municipio', 'nunique')
).reset_index()
# Criar dicionários separados para facilitar
zc_para_regiao = {zc: info['regiao'] for zc, info in zc_info_dict.items()}
zc_para_area = {zc: info['area'] for zc, info in zc_info_dict.items()}

# Adicionar as colunas ao DataFrame
resultado['Região predominante'] = resultado['ZC'].map(zc_para_regiao)
resultado['Área (km²)'] = resultado['ZC'].map(zc_para_area)

# Calcular a porcentagem (área da ZC / área total * 100)
area_total = resultado['Área (km²)'].sum()
resultado['Porcentagem (%)'] = (resultado['Área (km²)'] / area_total * 100).round(2)
resultado

,ZC,num_municipios,Região predominante,Área (km²),Porcentagem (%)
0,5901,4,South,5430.239964,2.72
1,6601,1,Central-South,221.128654,0.11
2,6701,17,Central-South,16002.214917,8.01
3,6801,110,Central-South and Metropolitan Region of Curitiba,75856.201950,37.98
4,6901,19,Southwest and South,9525.499533,4.77
5,7501,24,Northwest,7211.771281,3.61
6,7601,76,North and North Pioneer,23114.615047,11.57
7,7701,98,West and North Central,39302.645005,19.68
8,7801,44,West,17702.602408,8.86
9,7901,6,Coastal,5381.529571,2.69


In [ ]:
resultado.to_excel('infos_por_zc.xlsx')